# AgriMind Phase 2: Time-Series Price Forecasting & Model Comparison

This component addresses **Objective 2** of the AgriMind project: **testing and comparing three different recurrent neural network models (SimpleRNN, LSTM, and GRU)** to determine which is most effective at forecasting future agricultural supply prices.

We use the **U.S. Fertilizer Use and Prices** dataset (`fertilizeruse.xls` sheet `Table7`) to extract a clean historical price index for **Urea 44-46% nitrogen** from 1960 to 2013 (biannual measurements taken in April and September/March).

### 1. Import Dependencies
We import core data handling, evaluation, plotting, and deep learning modules.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("TensorFlow version:", tf.__version__)

### 2. Load and Clean Dataset
We load `Table7` from `fertilizeruse.xls` containing average farm prices in dollars per short ton, clean headers, handle forward-filling of Year fields, and convert prices to floats.

In [ ]:
# Load Table 7 from the local spreadsheet file
xls_path = "fertilizeruse.xls"
df = pd.read_excel(xls_path, sheet_name='Table7')

# Extract clean headers from row 1
headers = df.iloc[1].tolist()
headers[0] = 'Year'
headers[1] = 'Month'
headers[2] = 'Dummy'
df.columns = headers

# Slice rows containing actual data (rows 5 to 111)
df_clean = df.iloc[5:112].copy()

# Clean empty spacing in Year column and forward fill
df_clean['Year'] = df_clean['Year'].replace(r'^\s*$', np.nan, regex=True).ffill()
df_clean = df_clean.drop(columns=['Dummy'])

# Clean strings and cast all chemical columns to numeric types
for col in df_clean.columns:
    if col not in ['Year', 'Month']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print(f"Dataset cleaned. Total records: {len(df_clean)} rows.")
df_clean.head(10)

### 3. Preprocess Sequential Data
We extract **Urea 44-46% nitrogen** prices, scale them using a `MinMaxScaler` in the range `[0, 1]`, and split them into training (80%) and testing (20%) datasets. We create sequence windows of size `lookback = 4` (equivalent to 2 years of biannual measurements) to predict the next value.

In [ ]:
# Select target price series (Urea)
urea_prices = df_clean['Urea 44-46% nitrogen'].values.reshape(-1, 1).astype('float32')

# Scale features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(urea_prices)

# Split train/test (80% / 20%)
train_size = int(len(scaled_prices) * 0.8)
train_data = scaled_prices[:train_size]
test_data = scaled_prices[train_size - 4:] # pad train tail for test sequences

def create_sequences(data, lookback=4):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, lookback=4)
X_test, y_test = create_sequences(test_data, lookback=4)

# Reshape inputs for recurrent layers: [samples, time_steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

### 4. Build and Train SimpleRNN Model

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model_rnn = Sequential([
    SimpleRNN(32, input_shape=(4, 1), activation='tanh', return_sequences=False),
    Dense(1)
])

model_rnn.compile(optimizer='adam', loss='mean_squared_error')
print(model_rnn.summary())

history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=150,
    batch_size=4,
    validation_split=0.1,
    verbose=0
)
print("SimpleRNN training complete.")

### 5. Build and Train LSTM Model

In [ ]:
model_lstm = Sequential([
    LSTM(32, input_shape=(4, 1), activation='tanh', return_sequences=False),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')
print(model_lstm.summary())

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=150,
    batch_size=4,
    validation_split=0.1,
    verbose=0
)
print("LSTM training complete.")

### 6. Build and Train GRU Model

In [ ]:
model_gru = Sequential([
    GRU(32, input_shape=(4, 1), activation='tanh', return_sequences=False),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mean_squared_error')
print(model_gru.summary())

history_gru = model_gru.fit(
    X_train, y_train,
    epochs=150,
    batch_size=4,
    validation_split=0.1,
    verbose=0
)
print("GRU training complete.")

### 7. Evaluate and Compare Models
We make predictions on the test set, inverse scale them to dollars per ton, and calculate Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² metrics.

In [ ]:
# Predict and inverse scale
preds_rnn = model_rnn.predict(X_test)
preds_lstm = model_lstm.predict(X_test)
preds_gru = model_gru.predict(X_test)

actual_y = scaler.inverse_transform(y_test.reshape(-1, 1))
decoded_rnn = scaler.inverse_transform(preds_rnn)
decoded_lstm = scaler.inverse_transform(preds_lstm)
decoded_gru = scaler.inverse_transform(preds_gru)

# Compute evaluation metrics
def compute_metrics(actual, pred):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    return mae, rmse, r2

metrics_rnn = compute_metrics(actual_y, decoded_rnn)
metrics_lstm = compute_metrics(actual_y, decoded_lstm)
metrics_gru = compute_metrics(actual_y, decoded_gru)

# Build comparison table
metrics_data = {
    "Model": ["SimpleRNN", "LSTM", "GRU"],
    "MAE (USD/ton)": [metrics_rnn[0], metrics_lstm[0], metrics_gru[0]],
    "RMSE (USD/ton)": [metrics_rnn[1], metrics_lstm[1], metrics_gru[1]],
    "R² Score": [metrics_rnn[2], metrics_lstm[2], metrics_gru[2]]
}

comparison_df = pd.DataFrame(metrics_data)
comparison_df

### 8. Visualize Price Forecasts & Comparisons
We plot the forecasts against the actual prices, and display a bar chart comparing performance metrics.

In [ ]:
# Plot Forecast vs Actual
plt.figure(figsize=(12, 6))
plt.plot(actual_y, label='Actual Urea Price', color='black', linewidth=2.5, marker='o')
plt.plot(decoded_rnn, label='SimpleRNN Prediction', color='red', linestyle='--', marker='x')
plt.plot(decoded_lstm, label='LSTM Prediction', color='blue', linestyle='-.', marker='s')
plt.plot(decoded_gru, label='GRU Prediction', color='green', linestyle=':', marker='^')
plt.title('Urea Price Forecasting: Model Comparison (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Time Step (Biannual, 2003-2013)', fontsize=12)
plt.ylabel('Price (USD / Short Ton)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.show()

# Plot MAE Bar Chart
plt.figure(figsize=(8, 5))
colors = ['#ffafcc', '#a2d2ff', '#ccff33']
plt.bar(comparison_df['Model'], comparison_df['MAE (USD/ton)'], color=colors, edgecolor='black', width=0.5)
plt.title('Mean Absolute Error (MAE) Comparison', fontsize=14, fontweight='bold')
plt.ylabel('MAE (USD / Short Ton)', fontsize=12)
plt.grid(axis='y', alpha=0.3)
for i, val in enumerate(comparison_df['MAE (USD/ton)']):
    plt.text(i, val + 1, f"${val:.2f}", ha='center', fontweight='bold')
plt.show()

### 9. Save Trained Models
We save each compiled neural network model file so they can be integrated into the AgriMind advisory module.

In [ ]:
model_rnn.save("agrimind_rnn_price_model.h5")
model_lstm.save("agrimind_lstm_price_model.h5")
model_gru.save("agrimind_gru_price_model.h5")

print("All 3 model files successfully exported to workspace!")